# WQ Alpha101 因子全量测试

`py-alpha-lib` 官方示例里的 **WQ Alpha101**（101 个因子）已原样内置到 minibt：

| 项 | 值 |
|---|---|
| 前缀 | `alpha101_`（CoreFunc：`alpha101_alpha001` …） |
| 入口 | `self.kline.alpha101.alpha001()`（**数据源就是 IndFrame 自己**） |
| 图形 | 全部**副图**（`overlap=False`） |
| 纯时序因子 | desk 选择器可见（`NON_KLINE_INDICATORS` 之外的） |
| 横截面 / 缺外部数据 | 进 `NON_KLINE_INDICATORS`（desk 不显示），但**方法照样在**：自己准备正确的数据就能算，数据不对由 alpha 抛**原生错误** |

> 依赖可选包：`py -3.12 -m pip install py-alpha-lib`（Rust 实现）

本教程按「单标的 → 横截面 → 与上游比对 → 无未来函数」四步，把 101 个
因子全部跑一遍。

In [ ]:
import numpy as np
import pandas as pd

import minibt
from minibt import *
options.set_conversion_mode = "free"

import minibt.indicators._alpha_factors as factors
from minibt.indicators.alpha101 import Alpha101

names = [name for name in dir(Alpha101)
         if name.startswith("alpha") and name != "NON_KLINE_INDICATORS"
         and callable(getattr(Alpha101, name))]
print("minibt", minibt.__version__, "| WQ Alpha101 因子数", len(names),
      "| 纯时序", len(Alpha101.PURE_INDICATORS),
      "| NON_KLINE", len(Alpha101.NON_KLINE_INDICATORS))


## 0. 案例数据：内置的三个 K 线

| 名称 | 合约 | 周期 | 根数 |
|---|---|---|---|
| `LocalDatas.l2609_60` | DCE.l2609 | **1 分钟** | 10000 |
| `LocalDatas.pp2609_60` | DCE.pp2609 | **1 分钟** | 10000 |
| `LocalDatas.test` | DCE.v2601 | **5 分钟** | 8964 |

In [ ]:
DATASETS = (
    ("l2609_60", LocalDatas.l2609_60, 0),
    ("pp2609_60", LocalDatas.pp2609_60, 1),
    ("test", LocalDatas.test, 2),
)

for name, data, _security in DATASETS:
    frame = data.dataframe
    print(f"{name:<10} {frame['symbol'].iloc[0]:<12} "
          f"{int(frame['duration'].iloc[0]):>4}s  {len(frame):>5} 根  "
          f"{frame['datetime'].iloc[0]} → {frame['datetime'].iloc[-1]}")

# 单标的（纯时序因子）用 5 分钟那套做基准
kline = LocalDatas.test.kline


## 1. 单标的：纯时序因子逐个跑

`NON_KLINE_INDICATORS` 之外的因子只用本标的价量 → 任何 K 线都能算。

In [ ]:
skip = set(Alpha101.NON_KLINE_INDICATORS)
pure_names = [name for name in dir(Alpha101)
              if name.startswith("alpha") and name not in skip
              and callable(getattr(Alpha101, name))]
print(f"纯时序 {len(pure_names)} 个，前十：", pure_names[:10])

ok, empty, failed = [], [], []
for name in pure_names:
    try:
        values = np.asarray(getattr(getattr(kline, "alpha101"), name)(),
                            dtype=float)
    except Exception as error:                      # noqa: BLE001
        failed.append((name, f"{type(error).__name__}: {error}"[:70]))
        continue
    if len(values) != len(kline):
        failed.append((name, f"长度 {len(values)} != {len(kline)}"))
        continue
    (ok if np.isfinite(values).any() else empty).append(name)

print(f"有值 {len(ok)} | 恒 NaN {len(empty)} | 失败 {len(failed)}")
if empty:
    print("恒 NaN（上游递归 SMA 的 NaN 种子同款）：", empty)
for name, message in failed[:10]:
    print("  [FAIL]", name, message)


### 访问器路径 + 数据源就是 IndFrame 自己

In [ ]:
first = kline.alpha101.alpha001()
print("alpha001:", type(first).__name__, "点", len(first),
      "| 最后值", round(float(first.values[-1]), 4))
print("plotinfo: isplot =", first._plotinfo.isplot,
      "| overlap =", first._plotinfo.overlap)

# 换一套数据（5 分钟 → 1 分钟）因子值就跟着变 ✓
other = LocalDatas.l2609_60.kline
a = np.asarray(first, dtype=float)
b = np.asarray(other.alpha101.alpha001(), dtype=float)
print("不同数据 → 不同结果:", not np.allclose(np.nan_to_num(a[-50:]),
                                              np.nan_to_num(b[-50:])))


## 2. 横截面因子：用三套内置 K 线拼面板

原版因子是**多标的横截面**（`RANK` 跨标的排名、`INDNEUTRALIZE` 按行业中性化），
必须有 `securityid` / `tradetime` 这类面板列。做法：

1. 1 分钟的两套 **重采样成 5 分钟**（和新数据同周期）；
2. 各取最后 **800 根**，用 bar 序号当 `tradetime`（真实研究要用同一时间段 ✓）；
3. 拼成 `securityid = 0/1/2` 的面板，再补一列 `indclass`（演示行业中性化）。

In [ ]:
SIZE = 800


def to_panel_frame(data, security: int) -> pd.DataFrame:
    """内置 K 线 → 5 分钟面板（OHLCV 重采样 + securityid / tradetime ✓）"""
    frame = data.dataframe[["datetime", "open", "high", "low", "close",
                            "volume"]].copy()
    frame = frame.set_index("datetime")
    if int(data.dataframe["duration"].iloc[0]) < 300:
        frame = (frame.resample("5min")
                 .agg({"open": "first", "high": "max", "low": "min",
                       "close": "last", "volume": "sum"})
                 .dropna())
    frame = frame.tail(SIZE)
    frame["securityid"] = security
    frame["tradetime"] = np.arange(len(frame))
    return frame.reset_index()


panel = pd.concat([to_panel_frame(data, security)
                   for _, data, security in DATASETS], ignore_index=True)
panel = panel.rename(columns={"volume": "vol"})
panel["indclass"] = np.where(panel["securityid"] == 2, 201010, 101010)

print("面板行数", len(panel), "| 标的数", panel["securityid"].nunique(),
      "| 每个标的", panel["tradetime"].nunique())
print(panel.head(3).to_string(index=False))


### 横截面因子逐个跑（101 个全上）

用研究接口 `minibt.factor.panel(panel, "wq101", no)`。
缺外部数据（`cap` / 基准指数 / `MKT` / `SMB` / `HML`）的因子会抛**原生错误**
—— 这不是框架在拦，而是 alpha 自己说数据不够 ✓

In [ ]:
cross_ok, cross_failed = [], []
for no in range(1, 101 + 1):
    try:
        values = minibt.factor.panel(panel, "wq101", no)
        assert len(values) == len(panel)
        cross_ok.append(no)
    except Exception as error:                      # noqa: BLE001
        cross_failed.append((no, f"{type(error).__name__}: {error}"[:70]))

print(f"跑通 {len(cross_ok)} / 101 | 失败 {len(cross_failed)}")
for no, message in cross_failed[:10]:
    print(f"  #{no} {message}")


### 抽查：跨标的排名（RANK）真的分组

横截面因子在每个 `tradetime` 上对所有标的排名 → 三个标的会拿到不同值。

In [ ]:
sample = 101
values = minibt.factor.panel(panel, "wq101", sample)
labelled = panel.assign(factor=np.asarray(values, dtype=float))
labelled = labelled[labelled["tradetime"] == labelled["tradetime"].max()]
print(f"最后一个 bar 上 #{sample} 的取值：")
print(labelled[["securityid", "tradetime", "factor"]].to_string(index=False))


## 3. 和上游 `py-alpha-lib` 示例逐值比对

内置只是**搬运**，公式一个字都没改 ✓

In [ ]:
from alpha.context import Context, ExecContext

import minibt.indicators._alpha_factors.wq101 as upstream

frame = factors.factor_frame(kline)
sampled = sorted(Alpha101.PURE_INDICATORS)[:8]
context = ExecContext(frame.copy())
for no in sampled:
    meta = factors.FACTORS["wq101"][no]
    with Context(groups=1):
        expected = np.asarray(getattr(upstream, meta["func"])(context),
                              dtype=float)
    got = np.asarray(getattr(kline.alpha101, meta["method"])(), dtype=float)
    print(f"#{no:<4} 一致:", bool(np.allclose(got, expected, equal_nan=True)))


## 4. 没有偷看未来

把数据截断到前 60% 再算一遍 —— 重叠区间的值必须一模一样。

In [ ]:
short = kline.iloc[: int(len(kline) * 0.6)]
leaks = []
for no in sorted(Alpha101.PURE_INDICATORS)[:10]:
    meta = factors.FACTORS["wq101"][no]
    full = np.asarray(getattr(kline.alpha101, meta["method"])(), dtype=float)
    part = np.asarray(getattr(short.alpha101, meta["method"])(), dtype=float)
    if not np.allclose(full[: len(short)], part, equal_nan=True, atol=1e-9):
        leaks.append(no)
print("未来函数检查：", "全部通过 ✓" if not leaks else f"可疑 {leaks}")


## 5. 抽样作图（6 个纯时序因子）

In [ ]:
pure = sorted(Alpha101.PURE_INDICATORS)
print("纯时序编号：", pure[:20], "…" if len(pure) > 20 else "")

picks = pure[len(pure) // 2: len(pure) // 2 + 6]
print("抽样：", picks)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figure, axes = plt.subplots(len(picks), 1, figsize=(9, 2.1 * len(picks)),
                            sharex=True)
for axis, no in zip(axes, picks):
    values = np.asarray(getattr(kline.alpha101, f"alpha{no:03d}")(),
                        dtype=float)
    axis.plot(kline.index, values, linewidth=1)
    axis.set_title(f"WQ Alpha101 #{no}", fontsize=9)
    axis.grid(alpha=0.3)
figure.tight_layout()
print("画好了", len(picks), "个因子")


## 6. 公式速查

In [ ]:
for no in sorted(Alpha101.PURE_INDICATORS)[:5]:
    print(f"#{no}:", minibt.factor.formula("wq101", no))


## 小结

| 需求 | 用法 |
|---|---|
| 图表上挂因子 | 选择器 →「WQ Alpha101」（只列纯时序，全部副图） |
| 策略里算因子 | `self.data.alpha101.alpha002()` |
| 横截面 / 研究 | `minibt.factor.panel(panel, "wq101", 101)` |
| 查编号 / 公式 | `minibt.factor.numbers("wq101")` / `minibt.factor.formula("wq101", 101)` |

`NON_KLINE_INDICATORS` = 101 个里非纯时序的那些（desk 不显示，但方法可用）✓